### **Entrenamiento y Ajuste de Hiperparámetros: Estrategia de Fusión Tardía (*Late Fusion*)**

Para la combinación de extractores seleccionada que mayor rendimiento conjunto obtienen, se entrenan a continuación las arquitecturas multimodales resultantes de combinar dichos extractores mediante la estrategia de **fusión tardía**, llevando a cabo un ajuste de hiperparámetros.

- Recordamos la combinación de extractores ganadora seleccionada:

    1. **`ResNet` + `Wav2Vec` + `RoBERTa`**

La arquitectura multimodal para esta estrategia de fusión tardía cuenta con los siguientes componentes:

1. **Extractores seleccionados por cada modalidad** (ya indicados).
2. **Adaptadores** independientes por cada modalidad que mapeen dichos vectores de características extraídos a un espacio latente común normalizado (constituidos por una LSTM, en el caso de audio y vídeo, + capa lineal, además de aplicar normalización, ReLU y dropout). 
3. **Fusión Tardía**: Mediante distintas técnicas como: **voto mayoritario**, **promedio** o **regresión logística**.


Se realiza el mismo proceso que en Fusión Temprana para el ajuste de hiperparámetros, con el objetivo de evitar la explosión combinatoria:

Se realizan dos Grid Search secuenciales: 

* **FASE 1: GRID SEARCH de Ventanas de contexto**. Partiendo de la configuración básica para el resto de hiperparámetros (la predeterminada), vamos a realizar dos búsquedas en rejilla para todas las combinaciones posibles de ventanas de contexto de las modalidades: 

    * **Vídeo**: `32 frames` vs `16 frames`.
    * **Audio**: `11 segundos` vs `7 segundos`.
    * **Texto**: `64 tokens` vs `32 tokens`.

De esta fase, fijamos la **combinación de ventanas de contexto** con las que mayor F1-Score Macro alcanza el modelo, superando el umbral de Recall de 0.45 para la clase de Estrés. Ante empate o valores muy ajustados en F1-Score Macro, se fija aquella configuración que obtenga el mayor **Recall** para la clase positiva (Estrés).

* **FASE 2: GRID SEARCH para la Capacidad de la Arquitectura, Regularización y Aprendizaje**. Con las ventanas ya óptimas fijadas, se ajustará el tamaño de la red y su nivel de regularización mediante otro grid search:

    * **Red grande**: `proj_dim = 512` y `hidden_mlp = 128` vs **Red pequeña**:  `proj_dim = 256` y `hidden_mp = 64`.
    * **Dropout**: `0.5`(estándar) vs `0.3`(menor regularización, permite mayor flujo de información). 
    * **Learning Rate (`lr`)**: `1e-4` vs `5e-5` vs `1e-5`. Probamos con valores bajos ya que nuestro dataset es pequeño y desbalanceado, no conviene dar pasos grandes.




Los valores indicados para el ajuste por cada hiperparámetro se han extraído de la literatura (excepto las ventanas, extraídas del EDA). Se han limitado a una cantidad razonable para evitar la explosión combinatoria. 

Por otro lado, los hiperparámetros que hemos decidido **FIJAR** (NO los ajustamos), dejando los valores por defecto, son:
- `epochs` (**default=20**): Número máximo de epochs.
- `patience` (**default=5**): Paciencia para Early Stopping.
- `pos_weight_mult`(**default=1.0**): Multiplicador para el peso de la clase positiva.
- `batch_size` (**default=32**): Tamaño del batch. Debido a su relación directa con el Learning Rate, se ha decidido fijarlo con 32 y ajustar Learning Rate. 
- `weight_decay` (**default=1e-2**): Weight decay (Regularización L2). El valor por defecto (1e-2) es el valor óptimo indicado en la literatura para el optimizador AdamW. 
- `hidden_lstm`: Se calcula y fija de forma dinámica en los adaptadores como paso intermedio (proporcional a la entrada del backbone y al `proj_dim`) para evitar cuellos de botella en la compresión temporal.

Para estos, se dejan los valores por defecto. Con esto, evitamos la explosión combinatoria.  


Los modelos se entrenarán y ajustarán sobre:
- Particiones *train* y *dev* (para validación) del **Dataset Global Unificado (MELD + IEMOCAP)**.
- Particiones *train* y *dev* (para validación) del **Dataset Individual MELD (preprocesado)**.
- Particiones *train* y *dev* (para validación) del **Dataset Individual IEMOCAP (preprocesado)**.

Se obtendrá un modelo *ajustado* por cada uno de estos datasets, resultando en **9 arquitecturas multimodales** (con los extractores ya fijados previamente, y por cada una de las tres técnicas de fusión tardía).

In [ ]:
# Importamos las librerías necesarias:

import os
import json
import matplotlib.pyplot as plt

---

## **Dataset Global Unificado (MELD + IEMOCAP)**

* **Mediante voto mayoritario**:
    * **FASE 1: GRID SEARCH de Ventanas de contexto**

In [ ]:
#-------------------------- Ventanas de a ajustar ------------------------

VENTANAS_VIDEO = [32, 16]
VENTANAS_AUDIO = [11, 7]
VENTANAS_TEXTO = ['roberta64', 'roberta32']

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec'
lr = 1e-4
do = 0.5
h_mlp = 128
prj_dim = 512

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for video_frames in VENTANAS_VIDEO:
    for audio_len in VENTANAS_AUDIO:
        for text_model in VENTANAS_TEXTO:

            nombre_base = (f"global_late_voto_{VIDEO_BACKBONE}{video_frames}_{AUDIO_BACKBONE}{audio_len}s_{text_model}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode voto "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {video_frames} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {audio_len} "
                       f"--text {text_model} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

 * **FASE 2: GRID SEARCH para la Capacidad de la Arquitectura, Regularización y Aprendizaje**

In [ ]:
#-------------------------- Hiperparámetros a ajustar ------------------------

confg_arq = [(512,128), (256,64)]
LRS = [1e-4,5e-5,1e-5]
dropouts = [0.5, 0.3]

# ------------------------ Valores ya ajustados ------------------------

VENTANA_VIDEO = 
VENTANA_AUDIO = 
VENTANA_TEXTO = 'roberta__'

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec' 

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for prj_dim, h_mlp in confg_arq:
    for lr in LRS:
        for do in dropouts:

            nombre_base = (f"global_late_voto_{VIDEO_BACKBONE}{VENTANA_VIDEO}_{AUDIO_BACKBONE}{VENTANA_AUDIO}s_{VENTANA_TEXTO}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode voto "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {VENTANA_VIDEO} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {VENTANA_AUDIO} "
                       f"--text {VENTANA_TEXTO} "
                       f"--proj_dim {prj_dim} "
                       f"--hidden_mlp {h_mlp} "
                       f"--lr {lr} "
                       f"--dropout {do} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

In [ ]:
# ----------- MOSTRAMOS LOS RESULTADOS Y LA CURVA DE APRENDIZAJE DEL MEJOR MODELO -----------

if mejor_config is not None:

    print(f"\nMEJOR CONFIGURACIÓN GLOBAL LATE MEDIANTE VOTO: {mejor_config}")
    print(f"Mejor Val F1: {mejor_f1:.4f}")
    
    json_mejor = f"historial_estres_{mejor_config}.json"
    with open(json_mejor, 'r') as f:
        historial_mejor = json.load(f)

    epochs_range = range(1, len(historial_mejor['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Curva de Aprendizaje — Mejor Modelo Global Late Voto\n{mejor_config}", fontsize=10)

    # --- Subplot 1: Train Loss vs Val Loss ---
    axes[0].plot(epochs_range, historial_mejor['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(epochs_range, historial_mejor['val_loss'],label='Val Loss',   marker='o')
    axes[0].set_title('Pérdida (Loss)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    # --- Subplot 2: Val F1-Macro vs Val Recall Estrés ---
    axes[1].plot(epochs_range, historial_mejor['val_f1'], label='Val F1-Macro', marker='o')
    axes[1].plot(epochs_range, historial_mejor['val_recall_estres'], label='Val Recall Estrés', marker='s')
    axes[1].axhline(y=RECALL_ESTRES_MINIMO, color='red', linestyle='--', label=f'Umbral Recall ({RECALL_ESTRES_MINIMO})')
    axes[1].set_title('Métricas de Validación')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Valor')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f"Fig_curva_aprendizaje_mejor_modelo_global_late_voto.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No se encontró ningún modelo que superara el filtro de Recall mínimo.")

* **Mediante promedio**:
    * **FASE 1: GRID SEARCH de Ventanas de contexto**

In [ ]:
#-------------------------- Ventanas de a ajustar ------------------------

VENTANAS_VIDEO = [32, 16]
VENTANAS_AUDIO = [11, 7]
VENTANAS_TEXTO = ['roberta64', 'roberta32']

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec'
lr = 1e-4
do = 0.5
h_mlp = 128
prj_dim = 512

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for video_frames in VENTANAS_VIDEO:
    for audio_len in VENTANAS_AUDIO:
        for text_model in VENTANAS_TEXTO:

            nombre_base = (f"global_late_promedio_{VIDEO_BACKBONE}{video_frames}_{AUDIO_BACKBONE}{audio_len}s_{text_model}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode promedio "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {video_frames} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {audio_len} "
                       f"--text {text_model} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

* **FASE 2: GRID SEARCH para la Capacidad de la Arquitectura, Regularización y Aprendizaje**

In [ ]:
#-------------------------- Hiperparámetros a ajustar ------------------------

confg_arq = [(512,128), (256,64)]
LRS = [1e-4,5e-5,1e-5]
dropouts = [0.5, 0.3]

# ------------------------ Valores ya ajustados ------------------------

VENTANA_VIDEO = 
VENTANA_AUDIO = 
VENTANA_TEXTO = 'roberta__'

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec' 

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for prj_dim, h_mlp in confg_arq:
    for lr in LRS:
        for do in dropouts:

            nombre_base = (f"global_late_promedio_{VIDEO_BACKBONE}{VENTANA_VIDEO}_{AUDIO_BACKBONE}{VENTANA_AUDIO}s_{VENTANA_TEXTO}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode promedio "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {VENTANA_VIDEO} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {VENTANA_AUDIO} "
                       f"--text {VENTANA_TEXTO} "
                       f"--proj_dim {prj_dim} "
                       f"--hidden_mlp {h_mlp} "
                       f"--lr {lr} "
                       f"--dropout {do} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

In [ ]:
# ----------- MOSTRAMOS LOS RESULTADOS Y LA CURVA DE APRENDIZAJE DEL MEJOR MODELO -----------

if mejor_config is not None:

    print(f"\nMEJOR CONFIGURACIÓN GLOBAL LATE MEDIANTE PROMEDIO: {mejor_config}")
    print(f"Mejor Val F1: {mejor_f1:.4f}")
    
    json_mejor = f"historial_estres_{mejor_config}.json"
    with open(json_mejor, 'r') as f:
        historial_mejor = json.load(f)

    epochs_range = range(1, len(historial_mejor['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Curva de Aprendizaje — Mejor Modelo Global Late Promedio\n{mejor_config}", fontsize=10)

    # --- Subplot 1: Train Loss vs Val Loss ---
    axes[0].plot(epochs_range, historial_mejor['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(epochs_range, historial_mejor['val_loss'],label='Val Loss',   marker='o')
    axes[0].set_title('Pérdida (Loss)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    # --- Subplot 2: Val F1-Macro vs Val Recall Estrés ---
    axes[1].plot(epochs_range, historial_mejor['val_f1'], label='Val F1-Macro', marker='o')
    axes[1].plot(epochs_range, historial_mejor['val_recall_estres'], label='Val Recall Estrés', marker='s')
    axes[1].axhline(y=RECALL_ESTRES_MINIMO, color='red', linestyle='--', label=f'Umbral Recall ({RECALL_ESTRES_MINIMO})')
    axes[1].set_title('Métricas de Validación')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Valor')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f"Fig_curva_aprendizaje_mejor_modelo_global_late_promedio.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No se encontró ningún modelo que superara el filtro de Recall mínimo.")

* **Mediante regresión logística**:
    * **FASE 1: GRID SEARCH de Ventanas de contexto**

In [ ]:
#-------------------------- Ventanas de a ajustar ------------------------

VENTANAS_VIDEO = [32, 16]
VENTANAS_AUDIO = [11, 7]
VENTANAS_TEXTO = ['roberta64', 'roberta32']

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec'
lr = 1e-4
do = 0.5
h_mlp = 128
prj_dim = 512

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for video_frames in VENTANAS_VIDEO:
    for audio_len in VENTANAS_AUDIO:
        for text_model in VENTANAS_TEXTO:

            nombre_base = (f"global_late_logistica_{VIDEO_BACKBONE}{video_frames}_{AUDIO_BACKBONE}{audio_len}s_{text_model}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode logistica "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {video_frames} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {audio_len} "
                       f"--text {text_model} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

* **FASE 2: GRID SEARCH para la Capacidad de la Arquitectura, Regularización y Aprendizaje**

In [ ]:
#-------------------------- Hiperparámetros a ajustar ------------------------

confg_arq = [(512,128), (256,64)]
LRS = [1e-4,5e-5,1e-5]
dropouts = [0.5, 0.3]

# ------------------------ Valores ya ajustados ------------------------

VENTANA_VIDEO = 
VENTANA_AUDIO = 
VENTANA_TEXTO = 'roberta__'

# ----------------------- Configuración básica --------------------------

VIDEO_BACKBONE = 'resnet'
AUDIO_BACKBONE = 'wav2vec' 

RECALL_ESTRES_MINIMO = 0.45  # Filtro mínimo obligatorio para la clase estrés, si el modelo no alcanza este recall en estrés se descarta

mejor_f1 = 0.0
mejor_config = None

for prj_dim, h_mlp in confg_arq:
    for lr in LRS:
        for do in dropouts:

            nombre_base = (f"global_late_logistica_{VIDEO_BACKBONE}{VENTANA_VIDEO}_{AUDIO_BACKBONE}{VENTANA_AUDIO}s_{VENTANA_TEXTO}_p{prj_dim}_h{h_mlp}_lr{lr}_do{do}")
            pth_path = f"pesos_modelo_estres_{nombre_base}.pth"
            json_path = f"historial_estres_{nombre_base}.json"

            comando = (f"python train.py "
                       f"--train_dataset global "
                       f"--fusion late "
                       f"--late_mode logistica "
                       f"--video {VIDEO_BACKBONE} "
                       f"--video_frames {VENTANA_VIDEO} "
                       f"--audio {AUDIO_BACKBONE} "
                       f"--audio_len {VENTANA_AUDIO} "
                       f"--text {VENTANA_TEXTO} "
                       f"--proj_dim {prj_dim} "
                       f"--hidden_mlp {h_mlp} "
                       f"--lr {lr} "
                       f"--dropout {do} ")
                        
            os.system(comando)

            # Extraemos ambas métricas del historial:
            val_f1 = 0.0
            val_recall = 0.0
            
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    historial = json.load(f)
                
                # Buscamos el índice de la época con el mejor F1
                mejores_f1_lista = historial['val_f1']
                if mejores_f1_lista:
                    mejor_indice = mejores_f1_lista.index(max(mejores_f1_lista))
                    
                    val_f1 = historial['val_f1'][mejor_indice]
                    val_recall = historial['val_recall_estres'][mejor_indice]

            print(f"{nombre_base} --> F1: {val_f1:.4f} | Recall estrés: {val_recall:.4f}")

            # Doble criterio: filtro de recall + selección por F1-Macro
            supera_filtro = val_recall >= RECALL_ESTRES_MINIMO
            mejora_f1 = val_f1 > mejor_f1

            if supera_filtro and mejora_f1:
                # ACTUALIZAMOS EL MEJOR MODELO (eliminando el antiguo mejor modelo)
                if mejor_config is not None:
                    for archivo in [f"pesos_modelo_estres_{mejor_config}.pth", f"historial_estres_{mejor_config}.json"]:
                        if os.path.exists(archivo):
                            os.remove(archivo)

                mejor_f1 = val_f1
                mejor_config = nombre_base
                print(f"--> NUEVO MEJOR MODELO (F1={mejor_f1:.4f} | Recall={val_recall:.4f}): {nombre_base}")
            else :
                if not supera_filtro:
                    print(f"--> DESCARTADO: Recall estrés ({val_recall:.4f}) < umbral ({RECALL_ESTRES_MINIMO})")
                for archivo in [pth_path, json_path]:
                    if os.path.exists(archivo):
                        os.remove(archivo)

# Mostramos la configuración ganadora:
print(f"\nCONFIGURACIÓN GANADORA: {mejor_config}")
print(f"Mejor Val F1: {mejor_f1:.4f}")

In [ ]:
# ----------- MOSTRAMOS LOS RESULTADOS Y LA CURVA DE APRENDIZAJE DEL MEJOR MODELO -----------

if mejor_config is not None:

    print(f"\nMEJOR CONFIGURACIÓN GLOBAL LATE MEDIANTE REGRESIÓN LOGÍSTICA: {mejor_config}")
    print(f"Mejor Val F1: {mejor_f1:.4f}")
    
    json_mejor = f"historial_estres_{mejor_config}.json"
    with open(json_mejor, 'r') as f:
        historial_mejor = json.load(f)

    epochs_range = range(1, len(historial_mejor['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Curva de Aprendizaje — Mejor Modelo Global Late Regresión Logística\n{mejor_config}", fontsize=10)

    # --- Subplot 1: Train Loss vs Val Loss ---
    axes[0].plot(epochs_range, historial_mejor['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(epochs_range, historial_mejor['val_loss'],label='Val Loss',   marker='o')
    axes[0].set_title('Pérdida (Loss)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    # --- Subplot 2: Val F1-Macro vs Val Recall Estrés ---
    axes[1].plot(epochs_range, historial_mejor['val_f1'], label='Val F1-Macro', marker='o')
    axes[1].plot(epochs_range, historial_mejor['val_recall_estres'], label='Val Recall Estrés', marker='s')
    axes[1].axhline(y=RECALL_ESTRES_MINIMO, color='red', linestyle='--', label=f'Umbral Recall ({RECALL_ESTRES_MINIMO})')
    axes[1].set_title('Métricas de Validación')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Valor')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f"Fig_curva_aprendizaje_mejor_modelo_global_late_logistica.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No se encontró ningún modelo que superara el filtro de Recall mínimo.")